# Delivery ETA Correction — EDA & Baselines
This notebook is for exploration. Reusable pipeline logic lives in `src/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/food_delivery_orders_dataset.csv')
df.shape

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
completed = df[df['actual_delivery_time_minutes'].notna()].copy()
completed['order_timestamp'] = pd.to_datetime(completed['order_timestamp'])
completed = completed.sort_values('order_timestamp').reset_index(drop=True)
completed['eta_error'] = (
    completed['actual_delivery_time_minutes']
    - completed['estimated_delivery_time_minutes']
)
completed.shape

In [ ]:
completed['actual_delivery_time_minutes'].describe()

In [ ]:
completed['actual_delivery_time_minutes'].hist(bins=40)
plt.xlabel('Actual delivery time (minutes)')
plt.ylabel('Orders')
plt.title('Target distribution')
plt.show()

In [ ]:
n = len(completed)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train = completed.iloc[:train_end].copy()
val = completed.iloc[train_end:val_end].copy()
test = completed.iloc[val_end:].copy()
print(train.shape, val.shape, test.shape)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_mae = mean_absolute_error(
    test['actual_delivery_time_minutes'],
    test['estimated_delivery_time_minutes']
)
baseline_rmse = np.sqrt(mean_squared_error(
    test['actual_delivery_time_minutes'],
    test['estimated_delivery_time_minutes']
))
print('Baseline MAE:', baseline_mae)
print('Baseline RMSE:', baseline_rmse)

In [ ]:
mean_correction = train['eta_error'].mean()
test['mean_corrected_eta'] = test['estimated_delivery_time_minutes'] + mean_correction
print('Mean correction:', mean_correction)
print('Mean-corrected MAE:', mean_absolute_error(
    test['actual_delivery_time_minutes'], test['mean_corrected_eta']))
print('Mean-corrected RMSE:', np.sqrt(mean_squared_error(
    test['actual_delivery_time_minutes'], test['mean_corrected_eta'])))

## Next
Run `python -m src.data.prepare`, then `python -m src.models.train` to train the reproducible XGBoost correction model.